# Example: Trigger Rules and Scorecard Comparison

In this example, we implement realistic trigger rules — drawdown-based de-risking and turnover caps — and test how they affect the rebalancing engine's behavior. We then produce a comprehensive scorecard that compares the AI engine to Session 1's static min-variance baseline and the buy-and-hold benchmark.

> **By the end of this example, you will be able to:**
> * Implement and test drawdown-triggered de-risking and turnover-capped rebalancing
> * Compare engine behavior under different trigger rule configurations
> * Produce a scorecard comparing the AI rebalancing engine to the Session 1 static baseline

## Setup, Data and Prerequisites

In [ ]:
include("Include.jl");

Load the data from the previous example — market data, engine context, and benchmark wealth series.

In [ ]:
let
    # Load data from Example 1 -
    data = load(joinpath(_PATH_TO_DATA, "engine-run-data.jld2"));

    global price_matrix = data["price_matrix"];
    global lambda_series = data["lambda_series"];
    global market_prices = data["market_prices"];
    global gm_ema = data["gm_ema"];
    global tickers = data["tickers"];
    global sim_params = data["sim_params"];
    global context = data["context"];
    global buyhold_wealth = data["buyhold_wealth"];
    global riskfree_wealth = data["riskfree_wealth"];
    global engine_wealth_permissive = data["engine_wealth"];

    # Constants -
    global Δt = 1.0 / 252.0;
    global N_short = 21;
    global N_long = 63;
    global offset = N_short + N_long;
    global n_trading_days = 252;
    global B₀ = context.B;
    global K = length(tickers);

    println("Loaded engine data: $(K) tickers, $(n_trading_days) trading days")
end

___
## Task 1: Drawdown-Based De-Risking
We test the drawdown trigger at three thresholds: 10%, 15%, and 25%. When portfolio wealth drops below the threshold from its peak, the engine moves to 100% cash — a circuit breaker. We compare the wealth curves to see how aggressive vs. conservative thresholds affect performance.

> **What should you see?** Tighter drawdown limits (10%) will cause the engine to go to cash earlier and more often — protecting capital but potentially missing recoveries. Looser limits (25%) allow more volatility but capture more upside. The optimal threshold depends on risk tolerance.

In [ ]:
let
    drawdown_limits = [0.10, 0.15, 0.25];
    colors = [:red :orange :green];
    labels = ["DD ≤ 10%", "DD ≤ 15%", "DD ≤ 25%"];

    global dd_wealth_curves = Dict{Float64, Array{Float64,1}}();
    global dd_results_dict = Dict{Float64, Dict{Int, MyRebalancingResult}}();

    p = plot(size=(800, 450), title="Drawdown-Triggered De-Risking",
        xlabel="Trading Day (after warmup)", ylabel="Wealth (\$)", legend=:topleft)

    for (j, dd_limit) in enumerate(drawdown_limits)

        rules = build(MyTriggerRules, (
            max_drawdown = dd_limit,
            max_turnover = 1.0,
            rebalance_schedule = ones(Int, n_trading_days)
        ));

        results = run_rebalancing_engine(context, rules, lambda_series; offset=offset);
        wealth = compute_wealth_series(results, price_matrix, tickers; offset=offset);
        dd_wealth_curves[dd_limit] = wealth;
        dd_results_dict[dd_limit] = results;

        plot!(p, 1:length(wealth), wealth, label=labels[j], linewidth=2, color=colors[j])
    end

    # add buy-and-hold reference -
    plot!(p, 1:length(buyhold_wealth), buyhold_wealth, label="Buy-and-Hold", 
        linewidth=1.5, color=:gray50, linestyle=:dash)
    p
end

___
## Task 2: Turnover-Capped Rebalancing
We test turnover caps at 25%, 50%, and 100% (uncapped) with a moderate drawdown limit (15%). When the proposed trade exceeds the cap, trades are scaled down proportionally. This controls transaction costs without eliminating adaptability.

> **What should you see?** Lower turnover caps produce smoother, less reactive portfolios — lower trading costs but slower adaptation. Higher caps allow the engine to respond more aggressively to sentiment shifts. The preference weight (gamma) time series shows how the engine's allocation preferences evolve.

In [ ]:
let
    turnover_caps = [0.25, 0.50, 1.00];
    colors = [:steelblue :coral :green];
    labels = ["τ ≤ 25%", "τ ≤ 50%", "τ ≤ 100%"];

    p = plot(size=(800, 450), title="Turnover-Capped Rebalancing (DD limit = 15%)",
        xlabel="Trading Day (after warmup)", ylabel="Wealth (\$)", legend=:topleft)

    for (j, τ) in enumerate(turnover_caps)

        rules = build(MyTriggerRules, (
            max_drawdown = 0.15,
            max_turnover = τ,
            rebalance_schedule = ones(Int, n_trading_days)
        ));

        results = run_rebalancing_engine(context, rules, lambda_series; offset=offset);
        wealth = compute_wealth_series(results, price_matrix, tickers; offset=offset);

        plot!(p, 1:length(wealth), wealth, label=labels[j], linewidth=2, color=colors[j])
    end

    plot!(p, 1:length(buyhold_wealth), buyhold_wealth, label="Buy-and-Hold",
        linewidth=1.5, color=:gray50, linestyle=:dash)
    p
end

**Visualize:** Preference weights (γ) over time for the moderate configuration (DD ≤ 15%, τ ≤ 50%).

> **What should you see?** The gamma values show which assets the engine prefers at each time step. During bullish periods (negative lambda), high-beta assets should have higher gamma. During bearish periods, the Bond (negative beta) should be preferred.

In [ ]:
let
    # Run moderate configuration -
    rules = build(MyTriggerRules, (
        max_drawdown = 0.15, max_turnover = 0.50,
        rebalance_schedule = ones(Int, n_trading_days)
    ));
    results = run_rebalancing_engine(context, rules, lambda_series; offset=offset);

    # Extract gamma time series -
    gamma_matrix = zeros(n_trading_days + 1, K);
    for day in 0:n_trading_days
        if haskey(results, day) && isdefined(results[day], :gamma)
            gamma_matrix[day + 1, :] = results[day].gamma;
        end
    end

    colors = [:steelblue :coral :green :goldenrod :purple];
    p = plot(size=(800, 400), title="Preference Weights (γ) Over Time",
        xlabel="Trading Day (after warmup)", ylabel="γ", legend=:topright)
    for (k, ticker) in enumerate(tickers)
        plot!(p, 1:(n_trading_days + 1), gamma_matrix[:, k], label=ticker, 
            linewidth=1.5, color=colors[k])
    end
    hline!(p, [0.0], label="", linestyle=:dash, color=:black, alpha=0.3)
    p
end

___
## Task 3: Comprehensive Scorecard — Engine vs. Baselines
We produce a scorecard comparing three strategies: the AI rebalancing engine (moderate rules: DD ≤ 15%, τ ≤ 50%), Session 1's static min-variance buy-and-hold, and equal-weight buy-and-hold. The scorecard tracks return, volatility, Sharpe ratio, max drawdown, and effective trading cost.

> **What should you see?** The engine should show better risk-adjusted performance (lower drawdown, potentially better Sharpe) than static allocations, at the cost of higher turnover. The scorecard quantifies exactly how much adaptability costs and what it buys.

In [ ]:
let
    # Run the moderate engine configuration -
    rules = build(MyTriggerRules, (
        max_drawdown = 0.15, max_turnover = 0.50,
        rebalance_schedule = ones(Int, n_trading_days)
    ));
    results = run_rebalancing_engine(context, rules, lambda_series; offset=offset);
    engine_wealth = compute_wealth_series(results, price_matrix, tickers; offset=offset);

    # Helper: compute metrics from a wealth series -
    function scorecard_metrics(wealth::Array{Float64,1}, label::String)
        returns = diff(wealth) ./ wealth[1:end-1];
        total_return = (wealth[end] / wealth[1] - 1.0) * 100;
        annual_return = total_return; # already 1 year
        vol = std(returns) * sqrt(252) * 100;
        sharpe = vol > 0 ? annual_return / vol : 0.0;
        
        # max drawdown from wealth -
        peak = accumulate(max, wealth);
        dd = maximum((peak .- wealth) ./ peak) * 100;

        return (label, round(annual_return, digits=2), round(vol, digits=2),
            round(sharpe, digits=2), round(dd, digits=2))
    end

    # Compute metrics for each strategy -
    eng = scorecard_metrics(engine_wealth, "AI Engine (DD≤15%, τ≤50%)");
    bh = scorecard_metrics(buyhold_wealth, "Buy-and-Hold (equal-weight)");
    rf = scorecard_metrics(collect(riskfree_wealth), "Risk-Free (4.3%)");

    # Build scorecard table -
    scorecard = DataFrame(
        "Strategy" => [eng[1], bh[1], rf[1]],
        "Return (%)" => [eng[2], bh[2], rf[2]],
        "Volatility (%)" => [eng[3], bh[3], rf[3]],
        "Sharpe Ratio" => [eng[4], bh[4], rf[4]],
        "Max Drawdown (%)" => [eng[5], bh[5], rf[5]]
    );

    println("═"^65)
    println("  SESSION 2 SCORECARD: AI Engine vs. Baselines")
    println("═"^65)
    pretty_table(scorecard, tf=tf_markdown)

    # Save for Session 3 -
    save(joinpath(_PATH_TO_DATA, "session2-scorecard.jld2"),
        Dict("scorecard" => scorecard, "engine_wealth" => engine_wealth,
             "buyhold_wealth" => buyhold_wealth));
end

**Visualize:** Final wealth comparison — all three strategies side by side.

> **What should you see?** The engine's wealth curve should show the impact of trigger rules — flatter during de-risked periods, actively adapting otherwise. The scorecard numbers quantify the trade-off between adaptability and cost.

In [ ]:
let
    # Recompute moderate engine wealth -
    rules = build(MyTriggerRules, (
        max_drawdown = 0.15, max_turnover = 0.50,
        rebalance_schedule = ones(Int, n_trading_days)
    ));
    results = run_rebalancing_engine(context, rules, lambda_series; offset=offset);
    engine_wealth = compute_wealth_series(results, price_matrix, tickers; offset=offset);
    days = 1:length(engine_wealth);

    plot(days, engine_wealth, label="AI Engine (DD≤15%, τ≤50%)", linewidth=2.5, color=:steelblue)
    plot!(days, buyhold_wealth, label="Buy-and-Hold (equal-weight)", linewidth=2, color=:coral, linestyle=:dash)
    plot!(days, collect(riskfree_wealth), label="Risk-Free (4.3%)", linewidth=1.5, color=:gray50, linestyle=:dot)
    xlabel!("Trading Day (after warmup)")
    ylabel!("Wealth (\$)")
    title!("Session 2: AI Rebalancing Engine vs. Baselines")
    plot!(size=(800, 450), legend=:topleft)
end

___
## Summary
In this example, we tested the rebalancing engine under realistic trigger rules and produced a comprehensive scorecard:

- **Drawdown limits** act as circuit breakers — tighter limits protect capital but can miss recoveries
- **Turnover caps** control trading costs by limiting how much the portfolio can change per rebalance
- **The scorecard** quantifies the trade-off: the AI engine adapts to market conditions at the cost of turnover, while static strategies are cheaper but blind to regime changes

The scorecard and engine results are saved for Session 3, where we'll stress-test this engine against HMM-generated regime-switching market paths.

### Disclaimer
This content is for educational purposes only and does not constitute investment advice. The examples use synthetic data and simplified models.